In [7]:
import torch
import ultralytics

print("--- SYSTEM CHECK ---")
print(f"YOLO Version: {ultralytics.__version__}")
print(f"GPU Detected: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "WARNING: CPU ONLY")

--- SYSTEM CHECK ---
YOLO Version: 8.4.33
GPU Detected: NVIDIA GeForce RTX 4050 Laptop GPU


In [8]:
import json
import os
import yaml

os.makedirs('labels', exist_ok=True)
print("📖 Reading FathomNet Database...")

with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# Map categories 0 to 78
sorted_cats = sorted(data['categories'], key=lambda x: x['id'])
class_map = {cat['id']: i for i, cat in enumerate(sorted_cats)}
names_list = [cat['name'] for cat in sorted_cats]
images_dict = {img['id']: img for img in data['images']}

print("📝 Generating strict YOLO labels...")
count = 0
for ann in data['annotations']:
    img = images_dict.get(ann['image_id'])
    if not img: continue
    
    w, h = img['width'], img['height']
    bbox = ann['bbox'] 
    
    x_c = max(0, min((bbox[0] + bbox[2] / 2) / w, 1.0))
    y_c = max(0, min((bbox[1] + bbox[3] / 2) / h, 1.0))
    w_n = max(0, min(bbox[2] / w, 1.0))
    h_n = max(0, min(bbox[3] / h, 1.0))
    
    clean_filename = os.path.basename(img['file_name'])
    file_id = os.path.splitext(clean_filename)[0]
    label_path = os.path.join('labels', f"{file_id}.txt")
    yolo_class = class_map[ann['category_id']]
    
    with open(label_path, 'w') as f:
        f.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")
    count += 1

yaml_data = {
    'path': '.', 
    'train': 'images',
    'val': 'images',
    'nc': len(names_list),
    'names': names_list
}

with open('fathomnet.yaml', 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"✅ SUCCESS! Created {count} labels and updated fathomnet.yaml.")

📖 Reading FathomNet Database...
📝 Generating strict YOLO labels...
✅ SUCCESS! Created 23699 labels and updated fathomnet.yaml.


In [3]:
from ultralytics import YOLO

if __name__ == '__main__':
    # Start fresh with a blank Nano model
    model = YOLO('yolov8n.pt') 

    print("🚀 Initiating Deep Sea Training...")

    model.train(
        data='fathomnet.yaml',
        epochs=50,
        imgsz=640,
        batch=16, 
        device=0, 
        workers=0 # Set to 0 in Jupyter to prevent Windows multiprocessing crashes
    )

🚀 Initiating Deep Sea Training...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fathomnet.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

FileNotFoundError: [34m[1mtrain: [0mError loading data from C:\Users\azaed\OneDrive\Documents\FathomNet_Project\images
See https://docs.ultralytics.com/datasets for dataset formatting guidance.